In [ ]:
%load_ext autoreload
%autoreload 

In [ ]:
import pandas as pd
import numpy as np
from os import path, makedirs
from datetime import datetime

# local imports
import sys
sys.path.append('../../../')
from pyanalib.split_df_helpers import *
from analysis_village.cc1pi.systematics.final_variable_configs import VariableConfig
from analysis_village.cc1pi.systematics.utils import *
from analysis_village.cc1pi.systematics.constants import *
from pyanalib.covariance import *
from analysis_village.cc1pi.DataFrameUtils.DFLoading import *

from makedf.mcstat import get_MCstat_unc

# turn off PerformanceWarning 
# triggered by mismatched column levels
import warnings
warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)

In [ ]:
save_result = True
save_fig = save_result

save_fig_base_dir = "/exp/sbnd/data/users/lpelegri/syst/"

today_str = datetime.now().strftime("%Y%m%d")
save_fig_dir = path.join(save_fig_base_dir, "systematics-other-{}".format(today_str))

if save_fig:
    if not path.exists(save_fig_dir):
        makedirs(save_fig_dir)
    print("saving plots in ", save_fig_dir)

# Load df

In [ ]:
pot_weight_col = ('slc', 'wgt', '', '', '', '')

#Load CV dataframe
keys2load = ["cc1pi", "hdr", "histpotdf", "nudf"] ## keys from the configuration file
mc_bnb_df = load_df("/exp/sbnd/data/users/lpelegri/cafpyana_data/cc1pi_extended_syst.df", keys2load, 100)
mc_evt_df = mc_bnb_df['cc1pi']
mc_nu_df = mc_bnb_df['nudf']
mc_hdr_df = mc_bnb_df['hdr']

#Add weight column
data_tot_pot = 5.947e+18
mc_tot_pot = mc_hdr_df['pot'].sum()
print("mc_tot_pot: %.3e" %(mc_tot_pot))
mc_pot_scale = data_tot_pot / mc_tot_pot
print("mc_pot_scale: %.3e" %(mc_pot_scale))
mc_evt_df[pot_weight_col] = mc_pot_scale * np.ones(len(mc_evt_df))

#Do truth matchign
mc_evt_df = perform_truth_matching(mc_evt_df, mc_nu_df)
mc_nu_df[pot_weight_col] = mc_pot_scale * np.ones(len(mc_nu_df))

# Perform selection

In [ ]:
mc_obvious_cosmic_mask = mc_evt_df.slc.cut.obvious_cosmic
mc_t0_mask = mc_evt_df.slc.cut.t0
mc_is_inside_FV_mask = mc_evt_df.slc.cut.inside_FV
mc_nu_score_mask = mc_evt_df.slc.cut.nu_score
mc_track_mask = mc_evt_df.slc.cut.track
mc_shower_mask = mc_evt_df.slc.cut.shower 
mc_chi2_mask = mc_evt_df.slc.cut.MIP_candidates 
mc_angle_mask = mc_evt_df.slc.cut.angle 
mc_proton_BDT_mask = mc_evt_df.slc.cut.proton_BDT
mc_containment_mask = mc_evt_df.slc.cut.containment 
mc_michel_mask = mc_evt_df.slc.cut.michel 
mc_extra_pion_mask = mc_evt_df.slc.cut.extra_pion 
mc_energy_mask = mc_evt_df.slc.cut.energy

# 1. Define the order of cuts
mc_cut_sequence = [
    ("cosmic", mc_obvious_cosmic_mask),
    ("t0", mc_t0_mask),
    ("FV", mc_is_inside_FV_mask),
    ("nu_score", mc_nu_score_mask),
    ("track", mc_track_mask),
    ("chi2", mc_chi2_mask),
    ("shower", mc_shower_mask),
    ("angle", mc_angle_mask),
    ("proton_BDT", mc_proton_BDT_mask),
    ("containment", mc_containment_mask),
    ("michel", mc_michel_mask),
    ("extra_pion", mc_extra_pion_mask),
    ("energy", mc_energy_mask)
]

# 2. Build the cumulative masks
mc_cumulative_mak = None

for name, mask in mc_cut_sequence:
    if mc_cumulative_mak is None:
        mc_cumulative_mak = mask
    else:
        mc_cumulative_mak = mc_cumulative_mak & mask

In [ ]:
print(mc_evt_df.truth.nu_categ.value_counts())

In [ ]:
mc_evt_df = mc_evt_df[mc_cumulative_mak]

In [ ]:
#make it a slc df
mc_evt_df = (
        mc_evt_df
        .groupby(['__ntuple', 'entry', 'rec.slc..index'])
        .first()
    )
mc_evt_df = mc_evt_df.sort_index()


In [ ]:
print("==== breakdown of selected events ====")
print(mc_evt_df.truth.nu_categ.value_counts())
#print(mc_evt_df.genie_categ.value_counts())

In [ ]:

flux_systematics = [
    'expskin_Flux',
    'kzero_Flux',
    'horncurrent_Flux',
    'kminus_Flux',
    'kplus_Flux',
    'nucleoninexsec_Flux',
    'nucleonqexsec_Flux',
    'nucleontotxsec_Flux',
    'piminus_Flux',
    'pioninexsec_Flux',
    'pionqexsec_Flux',
    'piontotxsec_Flux',
    'piplus_Flux'
]

# Flux

In [ ]:
var_configs = [
    VariableConfig.all_evts(), 
    #VariableConfig.muon_momentum(),
    #VariableConfig.muon_direction(),
    VariableConfig.pion_momentum(),
    #VariableConfig.pion_direction(),
    #VariableConfig.angle_between_candidates(),
    #VariableConfig.num_protons(),
    #VariableConfig.delta_pt(),
    #VariableConfig.delta_alpha_T(),
    #VariableConfig.delta_phi_T()
]
show_plots = True
syst_dict = {}
#for cov_type in ["xsec", "rate"]:
for cov_type in ["rate"]:
    for var_config in var_configs:
        for syst_name in flux_systematics:
            univ_events, cv_events = get_univ_rates(cov_type, mc_evt_df, mc_nu_df, var_config, syst_name)
            ret_Flux = get_covariance_matrix(univ_events, cv_events)
            syst_dict[var_config.var_save_name + "_" + syst_name] = ret_Flux["cov_frac"]
    
    for var_config in var_configs:
        fig, ax = plt.subplots(figsize=(10,6))
    
        box = ax.get_position()
        ax.set_position([box.x0, box.y0, box.width * 0.8, box.height])
    
        frac_uncert_total = np.zeros(len(var_config.bin_centers))
    
        # Store everything first
        syst_storage = []
    
        for syst_name in flux_systematics:
            syst = syst_dict[var_config.var_save_name + "_" + syst_name]
            syst_uncert = np.sqrt(np.diag(syst))
            frac_uncert_total += syst_uncert ** 2
    
            values = syst_uncert * 1e2
            integral = np.sum(values)
    
            syst_storage.append((integral, syst_name, values))
    
        # Sort by integral (largest first)
        syst_storage.sort(key=lambda x: x[0], reverse=True)
    
        # Keep only top 5
        top5 = syst_storage[:5]
    
        handles = []
        labels = []
    
        # ---- Plot Top 5 ----
        for integral, syst_name, values in top5:
            h = ax.hist(
                var_config.bin_centers,
                bins=var_config.bins,
                weights=values,
                histtype="step",
                linewidth=2,
                label=syst_name
            )
            handles.append(h[2][0])
            labels.append(syst_name)
    
        # ---- Total ----
        frac_uncert_total = np.sqrt(frac_uncert_total)
        total_values = frac_uncert_total * 1e2
    
        total_handle = ax.hist(
            var_config.bin_centers,
            bins=var_config.bins,
            weights=total_values,
            histtype="step",
            linewidth=3,
            color="k",
            label="Total"
        )[2][0]
    
        handles.append(total_handle)
        labels.append("Total")
    
        # Legend
        ax.legend(
            handles,
            labels,
            loc="upper center",
            ncol=3,
            fontsize=11,
            frameon=True,
            edgecolor='gray'
        )
    
        ax.set_xlim(var_config.bins[0], var_config.bins[-1])
        ax.set_ylim(0, max(total_values) * 1.4)
        ax.set_xlabel(var_config.var_labels[1])
        ax.set_ylabel("Uncertainty [%]")
    
        ax.grid(which='major', linestyle='-', linewidth=0.7, alpha=0.7)
        ax.grid(which='minor', linestyle=':', linewidth=0.5, alpha=0.5)
        ax.minorticks_on()

# G4

In [ ]:
g4_systematics = [
    'reinteractions_kminus_Geant4',
    'reinteractions_kplus_Geant4',
    #'reinteractions_neutron_Geant4',
    'reinteractions_piminus_Geant4',
    'reinteractions_piplus_Geant4',
    'reinteractions_proton_Geant4'
]
label_map = {
    "reinteractions_kminus_Geant4": r"$K^{-}$",
    "reinteractions_kplus_Geant4": r"$K^{+}$",
    "reinteractions_neutron_Geant4": "n",
    "reinteractions_piminus_Geant4": r"$\pi^{-}$",  # Changed # to \
    "reinteractions_piplus_Geant4": r"$\pi^{+}$",   # Changed # to \
    "reinteractions_proton_Geant4": "p",
}

In [ ]:
var_configs = [
    #VariableConfig.all_evts(), 
    #VariableConfig.muon_momentum(),
    VariableConfig.muon_direction(),
    #VariableConfig.pion_momentum(),
    #VariableConfig.pion_direction(),
    #VariableConfig.angle_between_candidates(),
    #VariableConfig.num_protons(),
    #VariableConfig.delta_pt(),
    #VariableConfig.delta_alpha_T(),
    #VariableConfig.delta_phi_T()
]
show_plots = True
syst_dict = {}
#for cov_type in ["xsec"]:
for cov_type in ["xsec", "rate"]:
    for var_config in var_configs:
        for syst_name in g4_systematics:
            univ_events, cv_events = get_univ_rates(cov_type, mc_evt_df, mc_nu_df, var_config, syst_name)
            '''
            get_univ_rates(evtdf=mc_evt_df,
                                                var_config=var_config,
                                                n_univ=100,
                                                bkgd_subtract=True,
                                                syst_name=syst_name, 
                                                )
            '''
            
            ret_Flux = get_covariance_matrix(univ_events, cv_events)
            syst_dict[var_config.var_save_name + "_" + syst_name] = ret_Flux["cov_frac"]
      
    show_plots = True
    # Assuming syst_dict, var_configs, and g4_systematics are already defined

    for var_config in var_configs:
        # Use standard proportions to match the reference image
        fig, ax = plt.subplots(figsize=(8, 6))
        
        frac_uncert_total = np.zeros(len(var_config.bin_centers))
        
        # 1. Plot the dynamic systematics (G4, Genie, etc.)
        for syst_name in g4_systematics:
            # Retrieve the covariance matrix and calculate diagonal uncertainty
            syst = syst_dict[var_config.var_save_name + "_" + syst_name]
            syst_uncert = np.sqrt(np.diag(syst))
            frac_uncert_total += syst_uncert ** 2
            
            ax.hist(
                var_config.bin_centers,
                bins=var_config.bins,
                weights=syst_uncert * 1e2,
                histtype="step",
                linewidth=2,
                label=label_map[syst_name],
                zorder=3
            )
    
        # 2. Add 'flat' systematics (e.g., POT, Ntargets) if they are in your script
        # This matches the 'underneath' logic from your reference code
        # flat_systs = [pot_frac_unc, ntargets_frac_unc, nu_score]
        # flat_names = ["POT", "Ntargets", "NuScore"]
        # for name, val in zip(flat_names, flat_systs):
        #     syst_uncert = val * np.ones(len(var_config.bin_centers))
        #     frac_uncert_total += syst_uncert ** 2
        #     ax.hist(var_config.bin_centers, bins=var_config.bins, weights=syst_uncert*1e2, 
        #             histtype="step", linewidth=2, label=name, zorder=1)
    
        # 3. Calculate and plot the Total Uncertainty (Black line)
        frac_uncert_total = np.sqrt(frac_uncert_total)
        total_values = frac_uncert_total * 1e2
        
        ax.hist(
            var_config.bin_centers,
            bins=var_config.bins,
            weights=total_values,
            histtype="step",
            linewidth=2,
            color="k",
            label="Total",
            zorder=5  # Ensure the total is always on top
        )
    
        # --- LEGEND & FORMATTING TO MATCH IMAGE ---
        
        # ncol=3 creates the multi-column look
        # loc="upper center" places it at the top of the axes
        ax.legend(
            loc="upper center", 
            ncol=3, 
            fontsize=11, 
            frameon=True,
            edgecolor='gray'
        )
    
        # Set limits and labels
        ax.set_xlim(var_config.bins[0], var_config.bins[-1])
        # Multiplier 1.4 provides the 'white space' at the top for the legend
        ax.set_ylim(0, max(total_values) * 1.4) 
        
        ax.set_xlabel(var_config.var_labels[1])
        ax.set_ylabel("Uncertainty [%]")
        
        # Grid settings to match the crisp look of the plot
        ax.grid(which='major', linestyle='-', linewidth=0.7, alpha=0.7)
        ax.grid(which='minor', linestyle=':', linewidth=0.5, alpha=0.5)
        ax.minorticks_on()
    
        if show_plots:
            plt.show()

# GENIE

In [ ]:
genie_systematics_multisim = [
    'GENIEReWeight_SBN_v1_multisim_RPA_CCQE',
    #'GENIEReWeight_SBN_v1_multisim_CoulombCCQE',
    'GENIEReWeight_SBN_v1_multisim_NormCCMEC',
    #'GENIEReWeight_SBN_v1_multisim_NormNCMEC',
    #'GENIEReWeight_SBN_v1_multisim_RDecBR1gamma',
    'GENIEReWeight_SBN_v1_multisim_RDecBR1eta',
    #'GENIEReWeight_SBN_v1_multisim_NonRESBGvpCC1pi',
    #'GENIEReWeight_SBN_v1_multisim_NonRESBGvpCC2pi',
    #'#GENIEReWeight_SBN_v1_multisim_NonRESBGvpNC1pi',
    #'GENIEReWeight_SBN_v1_multisim_NonRESBGvpNC2pi',
    'GENIEReWeight_SBN_v1_multisim_NonRESBGvnCC1pi',
    'GENIEReWeight_SBN_v1_multisim_NonRESBGvnCC2pi',
    #'GENIEReWeight_SBN_v1_multisim_NonRESBGvnNC1pi',
    #'GENIEReWeight_SBN_v1_multisim_NonRESBGvnNC2pi',
    #'GENIEReWeight_SBN_v1_multisim_NonRESBGvbarpCC1pi',
    #'GENIEReWeight_SBN_v1_multisim_NonRESBGvbarpCC2pi',
    #'GENIEReWeight_SBN_v1_multisim_NonRESBGvbarpNC1pi',
    #'GENIEReWeight_SBN_v1_multisim_NonRESBGvbarpNC2pi',
    #'GENIEReWeight_SBN_v1_multisim_NonRESBGvbarnCC1pi',
    #'GENIEReWeight_SBN_v1_multisim_NonRESBGvbarnCC2pi',
    #'GENIEReWeight_SBN_v1_multisim_NonRESBGvbarnNC1pi',
    #'GENIEReWeight_SBN_v1_multisim_NonRESBGvbarnNC2pi',
]

genie_systematics_multisigma = [
    "GENIEReWeight_SBN_v1_multisigma_VecFFCCQEshape",
    'GENIEReWeight_SBN_v1_multisigma_ZExpA1CCQE',
    'GENIEReWeight_SBN_v1_multisigma_ZExpA2CCQE',
    'GENIEReWeight_SBN_v1_multisigma_ZExpA3CCQE',
    'GENIEReWeight_SBN_v1_multisigma_ZExpA4CCQE',
    "GENIEReWeight_SBN_v1_multisigma_DecayAngMEC",
    "GENIEReWeight_SBN_v1_multisigma_Theta_Delta2Npi",
    "GENIEReWeight_SBN_v1_multisigma_ThetaDelta2NRad",
    "GENIEReWeight_SBN_v1_multisigma_MaCCRES",
    "GENIEReWeight_SBN_v1_multisigma_MaNCRES",
    "GENIEReWeight_SBN_v1_multisigma_MvCCRES",
    "GENIEReWeight_SBN_v1_multisigma_MvNCRES",
    'GENIEReWeight_SBN_v1_multisigma_AhtBY',
    'GENIEReWeight_SBN_v1_multisigma_BhtBY',
    'GENIEReWeight_SBN_v1_multisigma_CV1uBY',
    'GENIEReWeight_SBN_v1_multisigma_CV2uBY',
    "GENIEReWeight_SBN_v1_multisigma_NormCCCOH", # Handled by re-tuning
    "GENIEReWeight_SBN_v1_multisigma_NormNCCOH",
    'GENIEReWeight_SBN_v1_multisigma_MFP_pi',
    'GENIEReWeight_SBN_v1_multisigma_FrCEx_pi',
    'GENIEReWeight_SBN_v1_multisigma_FrInel_pi',
    'GENIEReWeight_SBN_v1_multisigma_FrAbs_pi',
    'GENIEReWeight_SBN_v1_multisigma_FrPiProd_pi',
    'GENIEReWeight_SBN_v1_multisigma_MFP_N',
    'GENIEReWeight_SBN_v1_multisigma_FrCEx_N',
    'GENIEReWeight_SBN_v1_multisigma_FrInel_N',
    'GENIEReWeight_SBN_v1_multisigma_FrAbs_N',
    'GENIEReWeight_SBN_v1_multisigma_FrPiProd_N',
    'GENIEReWeight_SBN_v1_multisigma_MaNCEL',
    'GENIEReWeight_SBN_v1_multisigma_EtaNCEL',
]



In [ ]:
print(mc_nu_df.truth.GENIEReWeight_SBN_v1_multisigma_VecFFCCQEshape.columns)

In [ ]:
for syst in genie_systematics_multisigma:
    print("Checking:", syst)

    morph_key = ('truth', syst, 'morph', '', '', '')
    ps_key    = ('truth', syst, 'ps1', '', '', '')

    if morph_key in mc_nu_df.columns:
        print("  Found morph")
        s_morph = mc_nu_df[morph_key]
        for i in range(100):
            seed_input = str(i) + str(syst)
            np.random.seed(hash(seed_input) % (2**32))
            wgt = 1 + (s_morph - 1) * 2 * np.abs(np.random.normal(0, 1))
            mc_nu_df[('truth', syst, f'univ_{i}', '', '', '')] = wgt

    elif ps_key in mc_nu_df.columns:
        print("  Found ps1")
        s_ps = mc_nu_df[ps_key]
        for i in range(100):
            seed_input = str(i) + str(syst)
            np.random.seed(hash(seed_input) % (2**32))
            wgt = 1 + (s_ps - 1) * np.random.normal(0, 1)
            mc_nu_df[('truth', syst, f'univ_{i}', '', '', '')] = wgt



In [ ]:
missing_cols = mc_nu_df.columns.difference(mc_evt_df.columns)
mc_evt_df = mc_evt_df.join(mc_nu_df[missing_cols])

In [ ]:

show_plots = True
syst_dict = {}
for cov_type in ["xsec"]:
#for cov_type in ["xsec", "rate"]:
    for var_config in var_configs:
        for syst_name in genie_systematics_multisim:
            univ_events, cv_events = get_univ_rates(cov_type, mc_evt_df, mc_nu_df, var_config, syst_name)
            '''
            get_univ_rates(evtdf=mc_evt_df,
                                                var_config=var_config,
                                                n_univ=100,
                                                bkgd_subtract=True,
                                                syst_name=syst_name, 
                                                )
            '''
            
            ret_Flux = get_covariance_matrix(univ_events, cv_events)
            syst_dict[var_config.var_save_name + "_" + syst_name] = ret_Flux["cov_frac"]
      
    for var_config in var_configs:
        fig, ax = plt.subplots(figsize=(9,6))
    
        box = ax.get_position()
        ax.set_position([box.x0, box.y0, box.width * 0.8, box.height])
    
        frac_uncert_total = np.zeros(len(var_config.bin_centers))
    
        # Store everything first
        syst_storage = []
    
        for syst_name in genie_systematics_multisim:
            syst = syst_dict[var_config.var_save_name + "_" + syst_name]
            syst_uncert = np.sqrt(np.diag(syst))
            frac_uncert_total += syst_uncert ** 2
    
            values = syst_uncert * 1e2
            integral = np.sum(values)
    
            syst_storage.append((integral, syst_name, values))
    
        # Sort by integral (largest first)
        syst_storage.sort(key=lambda x: x[0], reverse=True)
    
        # Keep only top 5
        top5 = syst_storage[:5]
    
        handles = []
        labels = []
    
        # Plot only top 5
        for integral, syst_name, values in top5:
    
            h = ax.hist(
                var_config.bin_centers,
                bins=var_config.bins,
                weights=values,
                histtype="step",
                linewidth=2,
                #label=syst_name
            )
    
            handles.append(h[2][0])
            labels.append(syst_name)
    
        # ---- Total ----
        frac_uncert_total = np.sqrt(frac_uncert_total)
        total_values = frac_uncert_total * 1e2
    
        total_handle = ax.hist(
            var_config.bin_centers,
            bins=var_config.bins,
            weights=total_values,
            histtype="step",
            linewidth=3,
            color="k",
            #label="Total"
        )[2][0]
    
        handles.append(total_handle)
        labels.append("Total")
    
        # Legend
        '''
        ax.legend(
            loc="upper center", 
            ncol=2,
            fontsize=11, 
            frameon=True,
            edgecolor='gray'
        )
        '''
    
        ax.set_xlim(var_config.bins[0], var_config.bins[-1])
        ax.set_ylim(0, max(total_values) * 1.4)
        ax.set_xlabel(var_config.var_labels[1])
        ax.set_ylabel("Uncertainty [%]")
    
        ax.grid(which='major', linestyle='-', linewidth=0.7, alpha=0.7)
        ax.grid(which='minor', linestyle=':', linewidth=0.5, alpha=0.5)
        ax.minorticks_on()

In [ ]:
show_plots = True
syst_dict = {}
for cov_type in ["xsec"]:
#for cov_type in ["xsec", "rate"]:
    for var_config in var_configs:
        for syst_name in genie_systematics_multisigma:
            univ_events, cv_events = get_univ_rates(cov_type, mc_evt_df, mc_nu_df, var_config, syst_name)

            ret_Flux = get_covariance_matrix(univ_events, cv_events)
            syst_dict[var_config.var_save_name + "_" + syst_name] = ret_Flux["cov_frac"]
      
    for var_config in var_configs:
        fig, ax = plt.subplots(figsize=(10,6))
    
        box = ax.get_position()
        ax.set_position([box.x0, box.y0, box.width * 0.8, box.height])
    
        frac_uncert_total = np.zeros(len(var_config.bin_centers))
    
        # Store everything first
        syst_storage = []
    
        for syst_name in genie_systematics_multisigma:
            syst = syst_dict[var_config.var_save_name + "_" + syst_name]
            syst_uncert = np.sqrt(np.diag(syst))
            frac_uncert_total += syst_uncert ** 2
    
            values = syst_uncert * 1e2
            integral = np.sum(values)
    
            syst_storage.append((integral, syst_name, values))
    
        # Sort by integral (largest first)
        syst_storage.sort(key=lambda x: x[0], reverse=True)
    
        # Keep only top 5
        top5 = syst_storage[:5]
    
        handles = []
        labels = []
    
        # Plot only top 5
        for integral, syst_name, values in top5:
    
            h = ax.hist(
                var_config.bin_centers,
                bins=var_config.bins,
                weights=values,
                histtype="step",
                linewidth=2,
                label=syst_name
            )
    
            handles.append(h[2][0])
            labels.append(syst_name)
    
        # ---- Total ----
        frac_uncert_total = np.sqrt(frac_uncert_total)
        total_values = frac_uncert_total * 1e2
    
        total_handle = ax.hist(
            var_config.bin_centers,
            bins=var_config.bins,
            weights=total_values,
            histtype="step",
            linewidth=3,
            color="k",
            label="Total"
        )[2][0]
    
        handles.append(total_handle)
        labels.append("Total")
    
        # Legend
        ax.legend(
            loc="upper center", 
            ncol=3, 
            fontsize=11, 
            frameon=True,
            edgecolor='gray'
        )
    
        ax.set_xlim(var_config.bins[0], var_config.bins[-1])
        ax.set_ylim(0, max(total_values) * 1.4)
        ax.set_xlabel(var_config.var_labels[1])
        ax.set_ylabel("Uncertainty [%]")
    
        ax.grid(which='major', linestyle='-', linewidth=0.7, alpha=0.7)
        ax.grid(which='minor', linestyle=':', linewidth=0.5, alpha=0.5)
        ax.minorticks_on()

In [ ]:
def variation_hists_final_vars(evtdf=None, var_name=None, 
                    nevts_list=None,
                    datadf=None,
                    bins=None,
                    var_colors=None, var_labels=None,
                    plot_labels=["", "", ""],
                    vline = None,
                    textloc=[0.05, 0.55],
                    approval="internal",
                    plot=True,
                    save_fig=False, save_name=None): 

    bin_centers = 0.5 * (bins[:-1] + bins[1:])

    if evtdfs is not None:
        n_vars = len(evtdfs)
    elif nevts_list is not None:
        n_vars = len(nevts_list)
    else:
        raise ValueError("Either evtdfs or nevts_list must be provided")

    # get distribution from dfs
    if evtdfs is not None:
        vardfs, wgtdfs = [], []
        nevts_list = []
        mc_stat_err_list = []
        for df in evtdfs:
            vardf, wgtdf = get_clipped_evts(df, var_name, bins)
            vardfs.append(vardf)
            wgtdfs.append(wgtdf)


        # for sidx in range(n_vars):
            nevts, _ = np.histogram(vardf, bins=bins, weights=wgtdf)
            total_mc_err2, _ = np.histogram(vardf, bins=bins, weights=wgtdf**2)
            mc_stat_err = np.sqrt(total_mc_err2)
            nevts_list.append(nevts)
            mc_stat_err_list.append(mc_stat_err)

    if datadf is not None:
        vardf_data, _   = get_clipped_evts(datadf, var_name, bins)
        total_data, _ = np.histogram(vardf_data, bins=bins, weights=datadf.pot_weight)
        data_eylow, data_eyhigh = return_data_stat_err(total_data)

    # ===== plot =====
    fig, axs = plt.subplots(2, 1, figsize=(7.5, 7), 
                            sharex=True, gridspec_kw={'height_ratios': [4, 1]})

    fig.subplots_adjust(hspace=0.05)
    ax = axs[0]
    ax_r = axs[1]

    for sidx in range(n_vars):
        ax.hist(bin_centers,
                weights=nevts_list[sidx],
                bins=bins, 
                histtype="step" , 
                color=var_colors[sidx], 
                label=var_labels[sidx])

    if datadf is not None:
        ax.errorbar(bin_centers, 
                    total_data, 
                    yerr=np.vstack((data_eylow, data_eyhigh)),
                    color='black', 
                    fmt='o', markersize=5, capsize=3, linewidth=1.5,
                    label='Data')


    ax.set_ylabel(plot_labels[1])
    ax.set_title(plot_labels[2])
    ax.set_xlim(bins[0], bins[-1])
    ax.legend(loc="best")

    # ==== var/CV ratio panel
    for sidx in range(n_vars):
        if sidx == 0:
            continue
        # Avoid division by zero: ignore bins where denominator is 0
        ratio = np.full_like(nevts_list[0], np.nan, dtype=float)
        nonzero_mask = nevts_list[0] != 0
        ratio[nonzero_mask] = nevts_list[sidx][nonzero_mask] / nevts_list[0][nonzero_mask]
        # this_err = np.sqrt(
        #     (mc_stat_err_list[0] / nevts_list[0])**2 + 
        #     (mc_stat_err_list[sidx] / nevts_list[sidx])**2
        # )
        # Only plot nonzero, non-nan elements in the ratio
        valid_mask = (~np.isnan(ratio)) & (ratio != 0)
        ax_r.hist(bin_centers[valid_mask], bins=bins, weights=ratio[valid_mask], linewidth=1, histtype="step", color=var_colors[sidx])

    ax_r.axhline(1.0, color='red', linestyle='--', linewidth=1)
    ax_r.set_xlim(bins[0], bins[-1])
    ax_r.set_ylim(0.5, 1.5)
    ax_r.set_xlabel(plot_labels[0])
    ax_r.set_ylabel("Variation / CV")
    ax_r.grid(True)
    ax_r.minorticks_on()
    ax_r.grid(which='minor', linestyle=':', linewidth=0.5, color='gray', alpha=0.5)

    # ==== plot additions
    # vertical lines on main panel
    if vline is not None:
        for v in vline:
            ymax = ax.get_ylim()[1]
            ax.vlines(x=v, ymin=0, ymax=ymax*0.75, color='red', linestyle='--')

    # approval rank
    textloc_x, textloc_ha = get_textloc_x(nevts_list[0], bins, textloc)
    textloc_y = textloc[1]
    add_approval_text(approval, textloc_x, textloc_y, textloc_ha)

    # == save figure ==
    if save_fig:
        plt.savefig(save_name+fig_ext, bbox_inches='tight', dpi=dpi)

    if plot == True:
        plt.show()
    else:
        plt.close()

    return nevts_list
    

# All Uncertanties

In [ ]:
Total_Covariance_Frac = ret_Flux["cov_frac"] + ret_genie_rate["cov_frac"] + ret_MCstat["cov_frac"] + ret_G4["cov_frac"]

frac_unc_named_list = [
    (np.sqrt(np.diag(Total_Covariance_Frac)), "Total"),
    (np.sqrt(np.diag(ret_Flux["cov_frac"])), "Flux"),
    (np.sqrt(np.diag(ret_genie_xsec["cov_frac"])), "GENIE"),
    (np.sqrt(np.diag(ret_MCstat["cov_frac"])), "MC Stat"),
    (np.sqrt(np.diag(ret_G4["cov_frac"])), "G4"),
]

plot_frac_unc(frac_unc_named_list, var_config)